# FM Inference Demo
Loads the saved `fm_k8.npz` checkpoint from disk and makes predictions — **no retraining**.

Three things demonstrated:
1. `predict_pair` → logit, probability, predicted class
2. `explain_pair` → the four additive components of the FM score
3. Batch prediction over several drug pairs

In [ ]:
import json, sys, os
import numpy as np

# ── paths ─────────────────────────────────────────────────────────────────────
# Running from notebooks/ → parent is the repo root
ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA   = os.path.join(ROOT, 'data')
MODELS = os.path.join(ROOT, 'models', 'fm')
sys.path.insert(0, os.path.join(ROOT, 'src'))

from fm import FM, predict_pair, explain_pair, print_explanation
print('Imports OK')

In [ ]:
# ── 1. Load model and mapping from disk ───────────────────────────────────────
CKPT      = os.path.join(MODELS, 'fm_k8.npz')   # best k from validation experiment
THRESHOLD = 0.15                                  # val-tuned threshold

model      = FM.load(CKPT)                        # loads weights, no training
with open(os.path.join(DATA, 'drug_mapping.json')) as f:
    drug_to_id = json.load(f)['drug_to_id']

print(f'Loaded : {CKPT}')
print(f'k={model.k}   n_drugs={model.n_drugs}   threshold={THRESHOLD}')

In [ ]:
# ── 2. Predict one pair ───────────────────────────────────────────────────────
result = predict_pair(model, 'Dronedarone', 'Atenolol',
                      drug_to_id, threshold=THRESHOLD)

print(f"Drug 1          : {result['drug1']}")
print(f"Drug 2          : {result['drug2']}")
print(f"Raw logit       : {result['logit']:+.6f}")
print(f"Probability     : {result['probability']:.6f}")
print(f"Predicted class : {result['predicted_class']}  "
      f"({'Severe' if result['predicted_class'] else 'Not Severe'})")

In [ ]:
# ── 3. Explain the score — four additive terms ────────────────────────────────
#
#   z = b  +  w1[i]  +  w2[j]  +  dot(v1[i], v2[j])
#
expl = explain_pair(model, 'Dronedarone', 'Atenolol',
                    drug_to_id, threshold=THRESHOLD)
print_explanation(expl, threshold=THRESHOLD)

# Verify the sum by hand
total = expl['bias'] + expl['w1_term'] + expl['w2_term'] + expl['dot_product']
print(f"\nManual sum: {expl['bias']:+.6f} + {expl['w1_term']:+.6f} + "
      f"{expl['w2_term']:+.6f} + {expl['dot_product']:+.6f} = {total:+.6f}")
print(f"Matches logit:  {abs(total - expl['logit']) < 1e-10}")

In [ ]:
# ── 4. Batch predictions ─────────────────────────────────────────────────────
pairs = [
    ('Dronedarone',   'Atenolol'),
    ('Methadone',     'Sulfisoxazole'),
    ('Mirtazapine',   'Hydroxyzine'),
    ('Paliperidone',  'Chloroquine'),
    ('Acenocoumarol', 'Amiodarone'),
    ('Abiraterone',   'Acebutolol'),
]

print(f"{'Drug 1':<22} {'Drug 2':<22} {'Logit':>8}  {'P(Severe)':>10}  Prediction")
print('-' * 78)
for d1, d2 in pairs:
    r = predict_pair(model, d1, d2, drug_to_id, threshold=THRESHOLD)
    label = 'Severe' if r['predicted_class'] else 'Not Severe'
    print(f"{d1:<22} {d2:<22} {r['logit']:>+8.4f}  {r['probability']:>10.4f}  {label}")

In [ ]:
# ── 5. Sanity check: weights are identical to the saved file ──────────────────
saved = np.load(CKPT)
assert float(saved['b']) == model.b, 'Bias mismatch — something modified the model'
print(f'Checkpoint verified. bias = {model.b:+.8f}')
print('No retraining occurred — model is exactly as saved after experiment.')